# 04. 외식업 데이터셋

**목적:** 종로구·마포구 151개 상권 × 공개 상권자료의 외식업 10종 × 2023~2025년 12개 분기 패널을 만든다. `CS100001`~`CS100010` 전체를 결과를 보기 전에 고정하며 p값이나 매출 크기로 업종을 고르지 않는다.

01~03의 검증된 상권·공연 노출과 `jupyter-notebook/20261006/metadata/`의 서울시 고정 원자료를 사용한다. API 요청은 없으며 원자료를 바꾸지 않는다. 최종 입력은 `dataset/analysis/final/`, 보고서는 `reports/20261006/`에 저장한다.

In [ ]:
# 현재 구조 및 과거 고정 명세의 경로를 함께 해석합니다.
from pathlib import Path as _ProjectPath
import json as _bootstrap_json
_project_home = next(p for p in [_ProjectPath.cwd(), *_ProjectPath.cwd().parents]
                     if (p / 'environment/project_paths.ipynb').is_file())
_project_notebook = _bootstrap_json.loads((_project_home / 'environment/project_paths.ipynb').read_text(encoding='utf-8'))
for _project_cell in _project_notebook['cells']:
    if 'project-runtime' in _project_cell.get('metadata', {}).get('tags', []):
        exec(compile(''.join(_project_cell['source']), 'environment/project_paths.ipynb', 'exec'), globals())

from pathlib import Path
import hashlib
import json
import platform
from datetime import datetime, timezone
import importlib.metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = PROJECT_ROOT
if ROOT is None:
    raise FileNotFoundError('프로젝트 또는 notebooks 폴더에서 실행하세요.')
ANALYSIS, REPORTS = project_path('dataset/analysis/final'), project_path('reports/20261006')
FIGURES = REPORTS / 'figures'
for folder in [ANALYSIS, REPORTS, FIGURES]:
    folder.mkdir(parents=True, exist_ok=True)
YEARS = [2023, 2024, 2025]
QUARTERS = [f'{year}{q}' for year in YEARS for q in range(1, 5)]
KEY = ['quarter', 'area_id', 'industry_id']
AREA_QUARTER = ['area_id', 'quarter']
EXPOSURES = [f'known_performance_count_{r}m' for r in [300, 500, 800]]
pd.set_option('display.max_columns', 18)
pd.set_option('display.max_rows', 25)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')
print('Python:', platform.python_version(), '| 분석 단위: 상권 × 업종 × 분기')

## 1. 원천 추적과 재현 조건

`jupyter-notebook/20261006/metadata/seoul_analysis_sources.json`에서 검증한 연도별 매출·점포 CSV만 읽는다. 공연 노출은 02에서 계산하고 03에서 독립 검증한 결과다. 별도 보존된 `dataset/analysis/baseline_two_industry/`는 이전 계산과의 비교용 읽기 전용 기준선이다.

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

SEOUL_MANIFEST = project_path('jupyter-notebook/20261006/metadata/seoul_analysis_sources.json')
seoul_manifest = project_json_loads(SEOUL_MANIFEST.read_text(encoding='utf-8'))
source_csv_paths = {}
if seoul_manifest is not None:
    assert seoul_manifest['status'] == 'validated' and seoul_manifest['validation_passed']
    for item in seoul_manifest['items']:
        if item['dataset'] not in ['sales', 'stores']:
            continue
        candidates = [entry for entry in item['extracted'] if entry['path'].lower().endswith('.csv')]
        assert len(candidates) == 1, '연도별 CSV가 정확히 한 개여야 합니다.'
        entry = candidates[0]
        path = project_path(entry['path'])
        assert sha256_file(path) == entry['sha256'], f'재수집 원본 해시 불일치: {path.name}'
        source_csv_paths[(item['dataset'], int(item['year']))] = path
    assert len({p for p in source_csv_paths.values() if 'seoul_recollection' in p.as_posix()}) == 6
source_paths = list(source_csv_paths.values())
source_paths += [ANALYSIS / name for name in ['study_areas.csv',
    'sales_stores_panel.csv', 'performance_sales_panel.csv',
    'performance_facility_quarter.csv', 'venue_area_distances_800m.csv']]
source_paths += [REPORTS / 'linkage_analysis_summary.json', REPORTS / 'performance_link_methods.csv',
                 project_path('dataset/analysis/baseline_two_industry/snapshot_manifest.json')]
if SEOUL_MANIFEST.exists(): source_paths.append(SEOUL_MANIFEST)
source_hashes = pd.DataFrame([{'path': path.relative_to(ROOT).as_posix(),
    'bytes': path.stat().st_size, 'sha256': sha256_file(path)} for path in source_paths])
validation_sources = project_json_loads((project_path('jupyter-notebook/20261006/metadata/validation_sources.json')).read_text(encoding='utf-8'))
for record in validation_sources['files']:
    if record['role'] == 'baseline':
        assert sha256_file(project_path(record['path'])) == record['sha256'], record['path']
baseline_manifest = project_json_loads((project_path('dataset/analysis/baseline_two_industry/snapshot_manifest.json')).read_text(encoding='utf-8'))
legacy_raw_hashes = {(r['kind'], r['year']): r['sha256'] for r in baseline_manifest['legacy_raw_sources']}
source_revision_rows = [{'dataset': kind, 'year': year, 'path': path.relative_to(ROOT).as_posix(),
    'previous_sha256': legacy_raw_hashes[(kind, year)], 'current_sha256': sha256_file(path),
    'identical_to_baseline_raw': sha256_file(path) == legacy_raw_hashes[(kind, year)]}
    for (kind, year), path in source_csv_paths.items()]
source_revisions = pd.DataFrame(source_revision_rows)
all_raw_identical = bool(source_revisions['identical_to_baseline_raw'].all())
display(source_hashes[['path', 'bytes']])
display(source_revisions[['dataset', 'year', 'identical_to_baseline_raw']])

**해석:** 아래 산출물의 공연 노출 변수는 이 표의 재계산한 결합 패널 파일에 종속된다. 공연 목록과 상세 스냅샷의 시점 차이는 02번의 원천 명세로 확인한다. 원본 해시를 실행 마지막에 다시 확인한다.

## 2. 원자료의 모든 업종 코드·이름 확인

매출 원자료와 점포 원자료에 등장하는 **전체 업종**을 먼저 기록한다. 2025년 점포 자료만 영문 열 이름이므로 명시적으로 대응시킨다. 원자료 행과 결측을 유지하며 상권·분기·업종 키가 중복되거나 누락되면 중단한다.

In [ ]:
SALES_NAMES = {
    '기준_년분기_코드': 'quarter', '상권_코드': 'area_id',
    '서비스_업종_코드': 'industry_id', '서비스_업종_코드_명': 'industry',
    '당월_매출_금액': 'total_sales', '당월_매출_건수': 'sales_transactions',
    '시간대_21~24_매출_금액': 'night_sales', '주말_매출_금액': 'weekend_sales',
    '주중_매출_금액': 'weekday_sales'}
STORE_NAMES = {
    '기준_년분기_코드': 'quarter', '상권_코드': 'area_id',
    '서비스_업종_코드': 'industry_id', '서비스_업종_코드_명': 'industry',
    '점포_수': 'store_count', '유사_업종_점포_수': 'similar_industry_store_count',
    '개업_점포_수': 'opened_stores', '폐업_점포_수': 'closed_stores',
    '프랜차이즈_점포_수': 'franchise_stores',
    'stdr_yyqu_cd': 'quarter', 'trdar_cd': 'area_id',
    'svc_induty_cd': 'industry_id', 'svc_induty_cd_nm': 'industry',
    'stor_co': 'store_count', 'similr_induty_stor_co': 'similar_industry_store_count',
    'opbiz_stor_co': 'opened_stores', 'clsbiz_stor_co': 'closed_stores',
    'frc_stor_co': 'franchise_stores'}
frames = {'sales': [], 'stores': []}
source_quality_rows, catalog_rows = [], []

In [ ]:
for kind, rename in [('sales', SALES_NAMES), ('stores', STORE_NAMES)]:
    for year in YEARS:
        path = source_csv_paths[(kind, year)]
        raw = pd.read_csv(path, encoding='cp949', low_memory=False,
                          usecols=lambda column: column in rename)
        frame = raw.rename(columns=rename)
        required = list(dict.fromkeys(rename.values()))
        assert set(required).issubset(frame), f'{path.name}: 필수 열 누락'
        frame = frame[required].copy()
        for column in KEY:
            frame[column] = frame[column].astype('string').str.replace(r'\.0$', '', regex=True)
        assert not frame[KEY].isna().any().any(), f'{path.name}: 키 누락'
        assert not frame.duplicated(KEY).any(), f'{path.name}: 키 중복'
        assert set(frame['quarter']) == {f'{year}{q}' for q in range(1, 5)}
        assert frame['industry'].notna().all()
        labels = frame.groupby(['industry_id', 'industry'], observed=True).size()
        catalog_rows.extend([{'dataset': kind, 'year': year, 'industry_id': key[0],
                              'industry': key[1], 'source_rows': int(n)}
                             for key, n in labels.items()])
        source_quality_rows.append({'dataset': kind, 'year': year, 'rows': len(frame),
            'industries': frame['industry_id'].nunique(), 'duplicate_keys': 0,
            'missing_key_rows': 0, 'quarter_count': frame['quarter'].nunique()})
        frames[kind].append(frame)
source_catalog = pd.DataFrame(catalog_rows).sort_values(['industry_id', 'dataset', 'year'])
source_quality = pd.DataFrame(source_quality_rows)
sales = pd.concat(frames['sales'], ignore_index=True)
stores = pd.concat(frames['stores'], ignore_index=True)
display(source_quality)
store_identity = stores['similar_industry_store_count'].eq(stores['store_count'] + stores['franchise_stores'])
assert store_identity.all(), '전체점포 = 일반 + 프랜차이즈 원자료 검증 실패'
print('원자료 점포 항등식 검증:', len(stores), '행')


**해석:** 원자료 전체 업종의 목록과 연도별 행 수는 `reports/20261006/08_source_industry_catalog.csv`에 저장한다. 전체 목록과 공식 외식업 범위의 일관성을 각각 점검한다. 자료 제공 업종 수가 연도에 따라 다른 사실은 외식업 코드 변경과 구분한다.

## 3. 분석 결과를 보기 전에 외식업 10종 고정

공식 원자료의 `CS100001`~`CS100010`을 모두 포함한다. 명칭은 원자료 표기(예: `커피-음료`, `호프-간이주점`)를 보존한다. ‘상위 10개’라는 경제 규모 순위는 만들지 않는다.

In [ ]:
selected_ids = [f'CS100{number:03d}' for number in range(1, 11)]
food_source = source_catalog.loc[source_catalog['industry_id'].str.startswith('CS100')].copy()
assert set(food_source['industry_id']) == set(selected_ids)
assert food_source.groupby('industry_id')['industry'].nunique().eq(1).all()
assert food_source.groupby(['dataset', 'year'])['industry_id'].nunique().eq(10).all()
assert food_source.groupby('industry_id').size().eq(6).all()
industry_table = (food_source[['industry_id', 'industry']].drop_duplicates()
                  .sort_values('industry_id').reset_index(drop=True))
industry_table['selection_order'] = np.arange(1, len(industry_table) + 1)
industry_table['included'] = True
industry_table['selection_basis'] = 'all_official_CS100_food_services_before_model_results'
industry_table['stable_in_all_six_sources'] = True
label_check = source_catalog.groupby('industry_id', observed=True).agg(
    label_count=('industry', 'nunique'), source_year_pairs=('year', 'size'))
label_check['selected_food_industry'] = label_check.index.isin(selected_ids)
display(industry_table[['selection_order', 'industry_id', 'industry', 'stable_in_all_six_sources']])
print('전체 원자료 코드 수:', len(label_check), '| 코드에 연결된 이름이 2개 이상:', int(label_check['label_count'].gt(1).sum()))

**해석:** 한식·중식·일식·양식음식점, 제과점, 패스트푸드점, 치킨전문점, 분식전문점, 호프-간이주점, 커피-음료가 확정 대상이다. 카페와 음식점을 포함한 **서울시 자료에서 제공하는 외식업 전체**이며 현실의 모든 사업체·세부 업태를 빠짐없이 관측했다는 뜻은 아니다.

## 4. 151상권 × 10업종 × 12분기의 완전한 격자

기존 `study_areas.csv`의 상권 ID를 그대로 쓴다. 격자에 원자료를 왼쪽 결합하여 미제공 행도 남기고, ‘원자료 행 없음’과 ‘원자료 행은 있으나 값 누락’을 구분한다.

In [ ]:
areas = pd.read_csv(ANALYSIS / 'study_areas.csv', dtype={'area_id': str, 'district_code': str})
assert len(areas) == 151 and not areas['area_id'].duplicated().any()
assert set(areas['district_code']) == {'11110', '11440'}
quarters = pd.DataFrame({'quarter': QUARTERS})
panel = (areas.merge(industry_table[['industry_id', 'industry']], how='cross')
         .merge(quarters, how='cross'))
for kind, frame, present in [('sales', sales, 'sales_record_present'),
                              ('stores', stores, 'store_record_present')]:
    selected = frame.loc[frame['area_id'].isin(areas['area_id']) &
                         frame['industry_id'].isin(selected_ids)].drop(columns='industry').copy()
    selected[present] = True
    panel = panel.merge(selected, on=KEY, how='left', validate='one_to_one')
    panel[present] = panel[present].eq(True)
panel['availability'] = np.select(
    [panel['sales_record_present'] & panel['store_record_present'],
     ~panel['sales_record_present'] & panel['store_record_present'],
     panel['sales_record_present'] & ~panel['store_record_present']],
    ['both_present', 'sales_absent', 'stores_absent'], default='both_absent')
panel['sales_record_missing'] = ~panel['sales_record_present']
panel['store_record_missing'] = ~panel['store_record_present']
panel['total_sales_missing'] = panel['total_sales'].isna()
panel['store_count_missing'] = panel['store_count'].isna()
assert len(panel) == 151 * 10 * 12 and not panel.duplicated(KEY).any()
display(panel.groupby(['district', 'availability'], observed=True).size().rename('행 수').unstack(fill_value=0))

**해석:** 원자료에 없는 행을 0으로 채우지 않는다. 표의 0은 해당 상태의 행 개수다.

## 5. 점포 정의와 유효성 표식

`regular_store_count`는 일반점포, `franchise_store_count`는 프랜차이즈점포, `total_store_count`는 전체점포다. `store_count`, `franchise_stores`, `similar_industry_store_count`는 이전 입력과 정확 비교하기 위한 별칭이다. **전체 = 일반 + 프랜차이즈**를 원자료와 패널에서 검증한다.

매출양수·일반점포0은 프랜차이즈점포가 있으면 오류가 아니다. `positive_sales_zero_stores`는 전체점포0일 때만 표시한다. 매출 추정 모집단과 점포 모집단의 일치를 확인하지 못했으므로 점포당 매출은 계산하지 않는다. 야간 비중은 21~24시 매출/총매출이며 분모 양수·분자 유효 범위에서만 정의한다.

In [ ]:
# 원자료 항등식: 전체 = 일반 + 프랜차이즈. 기존 이름은 기준선 비교용 별칭입니다.
panel['regular_store_count'] = panel['store_count']
panel['franchise_store_count'] = panel['franchise_stores']
panel['total_store_count'] = panel['similar_industry_store_count']
observed_stores = panel['store_record_present']
assert panel.loc[observed_stores, 'total_store_count'].eq(
    panel.loc[observed_stores, 'regular_store_count'] + panel.loc[observed_stores, 'franchise_store_count']).all()
sales_amount_columns = ['total_sales', 'night_sales', 'weekend_sales', 'weekday_sales']
panel['negative_sales_flag'] = panel[sales_amount_columns].lt(0).any(axis=1)
panel['negative_store_flag'] = panel[['regular_store_count', 'franchise_store_count', 'total_store_count']].lt(0).any(axis=1)
panel['night_exceeds_total'] = panel['night_sales'].gt(panel['total_sales'])
panel['weekend_exceeds_total'] = panel['weekend_sales'].gt(panel['total_sales'])
panel['weekday_weekend_difference'] = panel['weekday_sales'] + panel['weekend_sales'] - panel['total_sales']
for component, output in [('night_sales', 'night_share'), ('weekend_sales', 'weekend_share')]:
    valid = panel['total_sales'].gt(0) & panel[component].ge(0) & panel[component].le(panel['total_sales'])
    panel[output] = (panel[component] / panel['total_sales']).where(valid)
    panel[f'{output}_valid'] = valid
panel['positive_sales_zero_regular_stores'] = panel['total_sales'].gt(0) & panel['regular_store_count'].eq(0)
panel['positive_sales_zero_stores'] = panel['total_sales'].gt(0) & panel['total_store_count'].eq(0)
panel['valid_sales_and_stores'] = (panel['total_sales'].ge(0) & panel['total_store_count'].ge(0) &
                                  ~panel['negative_sales_flag'] & ~panel['negative_store_flag'])
panel['year'] = panel['quarter'].str[:4].astype(int)
panel['quarter_number'] = panel['quarter'].str[-1].astype(int)
flag_columns = ['negative_sales_flag', 'negative_store_flag', 'night_exceeds_total',
                'weekend_exceeds_total', 'positive_sales_zero_stores']
flag_summary = panel.groupby(['industry_id', 'industry'], observed=True)[flag_columns].sum().reset_index()
display(flag_summary)

**해석:** 일반점포가 0인 행과 전체점포가 0인 행을 구분한다. 비중의 결측은 원값 누락·분모 0·범위 오류를 확인한다. 일반점포 별칭을 전체점포로 해석하지 않는다.

## 6. 상권·분기 공연 노출 결합

02의 두 업종에서 같은 상권·분기별 300m·500m·800m 노출이 정확히 일치하는지 확인한 뒤 10개 업종에 결합한다. 이는 기간이 겹치는 등록 공연 ID 수다. 업종 반복은 독립적인 공연 관측치를 늘리지 않는다.

In [ ]:
old_performance = pd.read_csv(ANALYSIS / 'performance_sales_panel.csv',
    dtype={'area_id': str, 'quarter': str, 'industry_id': str, 'district_code': str},
    float_precision='round_trip')
exposure_agreement = (old_performance.groupby(AREA_QUARTER, observed=True)[EXPOSURES]
                      .nunique(dropna=False))
assert exposure_agreement.eq(1).all().all(), '기존 두 업종의 공연 수가 다릅니다.'
exposures = old_performance[AREA_QUARTER + EXPOSURES].drop_duplicates(AREA_QUARTER)
assert len(exposures) == 151 * 12 and exposures[EXPOSURES].notna().all().all()
assert (exposures[EXPOSURES].ge(0) & exposures[EXPOSURES].mod(1).eq(0)).all().all()
assert exposures[EXPOSURES[0]].le(exposures[EXPOSURES[1]]).all()
assert exposures[EXPOSURES[1]].le(exposures[EXPOSURES[2]]).all()
panel = panel.merge(exposures, on=AREA_QUARTER, how='left', validate='many_to_one')
assert panel[EXPOSURES].notna().all().all()
assert panel.groupby(AREA_QUARTER)[EXPOSURES].nunique().eq(1).all().all()
linkage_summary = project_json_loads((REPORTS / 'linkage_analysis_summary.json').read_text(encoding='utf-8'))
link_methods = pd.read_csv(REPORTS / 'performance_link_methods.csv')
display(link_methods)
display(exposures[EXPOSURES].describe().round(1))

In [ ]:
# 실제 회귀 노출에 기여하는 공연의 연결 근거를 반경별로 확인한다.
quarter_links = pd.read_csv(ANALYSIS / 'performance_facility_quarter.csv', dtype=str)
distances = pd.read_csv(ANALYSIS / 'venue_area_distances_800m.csv', dtype={'area_id': str, 'mt10id': str})
assert quarter_links.groupby('mt20id')[['mt10id', 'link_method']].nunique().eq(1).all().all()
unique_performances = quarter_links[['mt20id', 'mt10id', 'link_method']].drop_duplicates()
linkage_coverage_rows = []
for radius in [300, 500, 800]:
    facilities_in_radius = set(distances.loc[distances['distance_m'].le(radius), 'mt10id'])
    contributing = unique_performances.loc[unique_performances['mt10id'].isin(facilities_in_radius)]
    official_count = int(contributing['link_method'].eq('official_detail').sum())
    provisional_count = int(contributing['link_method'].isin(['exact_unique_name', 'prefix_unique_name']).sum())
    assert official_count + provisional_count == len(contributing)
    linkage_coverage_rows.append({'radius_m': radius, 'contributing_performance_ids': len(contributing),
        'official_ids': official_count, 'provisional_ids': provisional_count,
        'official_share_pct': 100 * official_count / len(contributing) if len(contributing) else None})
exposure_linkage_coverage = pd.DataFrame(linkage_coverage_rows)
exposure_linkage_coverage.to_csv(REPORTS / '08_exposure_linkage_coverage.csv', index=False, encoding='utf-8-sig')
display(exposure_linkage_coverage)

display(Markdown(
    f"**공연 스냅샷:** 수집 상태는 `{linkage_summary['collection_status']}`이며, "
    f"상세 확보 공연은 {linkage_summary['performance_detail_rows']:,}개입니다. "
    f"보관된 서울 공연 목록 {linkage_summary['performance_ids']:,}개 중 "
    f"공식 시설 ID 연결은 {linkage_summary['official_link_ids']:,}개, "
    f"이름 임시 연결은 {linkage_summary['provisional_name_link_ids']:,}개입니다. "
    f"좌표 연결 가능 공연은 {linkage_summary['performance_ids_with_coordinates']:,}개입니다. "
    '수집 완료 여부는 지정된 후보 목록을 기준으로 하며 서울 전체 목록의 상세 수집 완료를 뜻하지 않습니다. '
    '좌표나 유효한 기간이 없는 공연은 공간·분기 집계에 포함하지 않습니다. '
    '현재 상권 경계와 시설 위치를 과거 12분기에 공통 적용하며, '
    '공연 노출값을 업종마다 반복했다고 독립적인 공연 관측치가 늘어나는 것은 아닙니다.'))

## 7. 자료 제공률과 점포 규모

분모는 업종당 151상권 × 12분기 = 1,812행이다. 제공률은 원자료 행과 실제 값의 제공을 나눠 제시한다. 평균·중앙 점포 수는 점포 수가 제공된 상권·분기에서 계산하며 미제공 행을 0으로 포함하지 않는다. 상권 수는 적어도 한 분기 매출이 제공된 상권과 12분기 모두 제공된 상권을 나눠 센다.
표의 평균·중앙 점포 수와 점포 관측률은 **전체점포(total_store_count)** 기준이다.

In [ ]:
def describe_coverage(group):
    sales_by_area = group.groupby('area_id', observed=True)['total_sales'].count()
    stores_by_area = group.groupby('area_id', observed=True)['store_count'].count()
    both_by_area = group.assign(both=group['total_sales'].notna() & group['total_store_count'].notna()).groupby('area_id')['both'].sum()
    return pd.Series({
        'expected_rows': len(group), 'expected_areas': group['area_id'].nunique(),
        'sales_record_rows': int(group['sales_record_present'].sum()),
        'sales_value_rows': int(group['total_sales'].notna().sum()),
        'store_record_rows': int(group['store_record_present'].sum()),
        'store_value_rows': int(group['total_store_count'].notna().sum()),
        'sales_record_coverage_pct': group['sales_record_present'].mean() * 100,
        'sales_value_coverage_pct': group['total_sales'].notna().mean() * 100,
        'store_value_coverage_pct': group['total_store_count'].notna().mean() * 100,
        'observed_sales_areas': int(sales_by_area.gt(0).sum()),
        'complete_12q_sales_areas': int(sales_by_area.eq(12).sum()),
        'complete_12q_store_areas': int(stores_by_area.eq(12).sum()),
        'complete_12q_both_areas': int(both_by_area.eq(12).sum()),
        'mean_observed_store_count': group['total_store_count'].mean(),
        'median_observed_store_count': group['total_store_count'].median(),
        'explicit_zero_sales_rows': int(group['total_sales'].eq(0).sum()),
        'explicit_zero_store_rows': int(group['total_store_count'].eq(0).sum())})

coverage_rows, district_rows = [], []
for (industry_id, industry), group in panel.groupby(['industry_id', 'industry'], observed=True):
    coverage_rows.append(dict(industry_id=industry_id, industry=industry, **describe_coverage(group).to_dict()))
for (district, industry_id, industry), group in panel.groupby(['district', 'industry_id', 'industry'], observed=True):
    district_rows.append(dict(district=district, industry_id=industry_id, industry=industry, **describe_coverage(group).to_dict()))
coverage = pd.DataFrame(coverage_rows)
district_coverage = pd.DataFrame(district_rows)
catalog = industry_table.merge(coverage, on=['industry_id', 'industry'], validate='one_to_one')
catalog['data_availability_rank'] = catalog['sales_value_rows'].rank(method='min', ascending=False).astype(int)
display(catalog[['industry', 'sales_value_rows', 'sales_value_coverage_pct', 'observed_sales_areas',
                 'complete_12q_sales_areas', 'mean_observed_store_count', 'median_observed_store_count']])

In [ ]:
quarter_coverage = panel.groupby(['district', 'industry_id', 'industry', 'quarter'], observed=True).agg(
    expected_rows=('area_id', 'size'), sales_record_rows=('sales_record_present', 'sum'),
    sales_value_rows=('total_sales', 'count'), store_record_rows=('store_record_present', 'sum'),
    store_value_rows=('total_store_count', 'count')).reset_index()
quarter_coverage['sales_value_coverage_pct'] = quarter_coverage['sales_value_rows'] / quarter_coverage['expected_rows'] * 100
quarter_coverage['store_value_coverage_pct'] = quarter_coverage['store_value_rows'] / quarter_coverage['expected_rows'] * 100
display(district_coverage[['district', 'industry', 'sales_value_coverage_pct', 'store_value_coverage_pct',
                           'complete_12q_sales_areas', 'mean_observed_store_count']])

## 8. 점포 규모와 매출 관측 표본

전체점포 규모에 따른 매출 제공률과 외식업 10종이 모두 관측되는 상권·분기를 확인한다. 작은 점포 규모의 매출 미제공은 무작위 결측으로 가정하지 않는다. 업종별 서로 다른 관측 표본과 공통표본은 서로 다른 범위를 대표한다.

In [ ]:
panel['total_store_size_group'] = pd.cut(panel['total_store_count'], [-1, 0, 2, 5, 10, 50, np.inf],
    labels=['0', '1~2', '3~5', '6~10', '11~50', '51+']).astype('string').fillna('미제공')
store_size_coverage = panel.groupby('total_store_size_group', observed=True).agg(
    rows=('quarter', 'size'), sales_observed=('total_sales', 'count'))
store_size_coverage['sales_missing'] = store_size_coverage['rows'] - store_size_coverage['sales_observed']
store_size_coverage['sales_coverage_pct'] = store_size_coverage['sales_observed'] / store_size_coverage['rows'] * 100
common_counts = panel.groupby(AREA_QUARTER, observed=True)['total_sales'].count()
common_sample_summary = pd.DataFrame([{'all_10_industries_observed_area_quarters': int(common_counts.eq(10).sum()),
    'total_area_quarters': len(common_counts), 'regular_zero_positive_sales_rows': int(panel['positive_sales_zero_regular_stores'].sum()),
    'total_zero_positive_sales_rows': int(panel['positive_sales_zero_stores'].sum())}])
store_size_coverage.to_csv(REPORTS / '08_store_size_sales_coverage.csv', encoding='utf-8-sig')
common_sample_summary.to_csv(REPORTS / '08_common_sample_scope.csv', index=False, encoding='utf-8-sig')
display(store_size_coverage)
display(common_sample_summary)


**해석:** 전체점포 1~2개인 행의 매출 관측률과 미제공 행 수를 확인한다. 매출 미제공은 0원 매출이 아니다. 공통표본 분석은 업종 간 표본 차이를 줄이지만 자료가 많이 관측되는 상권 쪽으로 범위가 좁아진다. 실제 모형의 공통표본은 추가 유효성 조건 적용 후 05에서 다시 계산한다.
이 표의 171개 상권·분기는 관측 여부 기준이다. 회귀에서 상권당 2개 이상 분기 조건을 적용한 공통표본 170개 상권·분기(1,700행)와 분모가 다르다.

**해석:** `data_availability_rank`는 사용할 수 있는 매출 관측행 수의 순위이며 경제 규모 순위가 아니다. 상권 경계가 겹칠 수 있으므로 업종별 매출 단순 합계를 지역의 독립적인 총매출로 부르지 않는다. 관측률이 낮은 업종을 제외하거나 유의한 업종만 남기지 않으며 이후 모형에서 표본 수와 결측을 함께 보고한다.

## 9. 업종·지역별 제공률 시각화

왼쪽은 매출 값, 오른쪽은 점포 값의 제공률이다. 표본 격자를 분모로 쓰고 0~100%의 같은 척도를 유지한다.

In [ ]:
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
fig, axes = plt.subplots(1, 2, figsize=(12, 6.0), constrained_layout=True)
labels = industry_table['industry'].tolist()
for ax, value, title in zip(axes,
        ['sales_value_coverage_pct', 'store_value_coverage_pct'], ['매출 값 제공률', '점포 값 제공률']):
    table = district_coverage.pivot(index='industry', columns='district', values=value).reindex(labels)
    table = table.reindex(columns=['종로구', '마포구'])
    heat = ax.imshow(table.to_numpy(), vmin=0, vmax=100, cmap='Blues', aspect='auto')
    ax.set_xticks([0, 1], table.columns)
    ax.set_yticks(range(len(labels)), labels)
    ax.set_title(title, fontweight='bold')
    for row in range(len(labels)):
        for column in range(2):
            val = table.iloc[row, column]
            ax.text(column, row, f'{val:.1f}%', ha='center', va='center',
                    color='white' if val >= 65 else '#182536')
fig.colorbar(heat, ax=axes, label='151상권 × 12분기 격자 중 해당 지역의 제공률 (%)', shrink=0.8)
fig.suptitle('공식 외식업 10종 · 2023~2025년 자료 제공률', fontsize=15, fontweight='bold')
fig.savefig(FIGURES / '08_food_industry_coverage.png', dpi=180, bbox_inches='tight')
plt.show()

**해석:** 제공률 차이는 업종의 실제 영업 규모와 별개로 자료의 관측 가능성이 다르다는 신호다. 같은 업종에서도 지역별 관측 가능성이 다르면 지역 비교 모형의 표본 구성이 달라질 수 있다.

## 10. 기존 한식·커피 패널과 개정 여부 비교

동결한 기존 패널의 3,624행에 대해 키, 상권·업종 이름, 10개 원자료 수치 열, 자료 존재 표식을 정렬 후 정확 비교한다. 파생 비율은 CSV 소수점 직렬화 오차만 허용한다. 공연 노출값도 기존 두 업종에 대해 정확히 비교한다.

원본 6개의 해시가 모두 같으면 값의 정확한 일치를 요구한다. 원본이 개정되었으면 차이를 보고하며, 새 값을 과거 값으로 되돌리지 않는다.

In [ ]:
old_panel = pd.read_csv(project_path('dataset/analysis/baseline_two_industry/sales_stores_panel.csv'),
    dtype={'area_id': str, 'quarter': str, 'industry_id': str, 'district_code': str},
    float_precision='round_trip').sort_values(KEY).reset_index(drop=True)
legacy_ids = ['CS100001', 'CS100010']
legacy_new = panel.loc[panel['industry_id'].isin(legacy_ids)].sort_values(KEY).reset_index(drop=True)
raw_value_columns = ['total_sales', 'sales_transactions', 'night_sales', 'weekend_sales', 'weekday_sales',
                     'store_count', 'similar_industry_store_count', 'opened_stores', 'closed_stores', 'franchise_stores']
exact_columns = KEY + ['area_name', 'district_code', 'district', 'industry'] + raw_value_columns + [
    'sales_record_present', 'store_record_present', 'availability']
assert len(legacy_new) == len(old_panel) == 3624
pd.testing.assert_frame_equal(legacy_new[KEY], old_panel[KEY], check_dtype=False, check_exact=True)
legacy_comparison_rows = []
for column in exact_columns:
    mismatch = ~(legacy_new[column].eq(old_panel[column]) |
                 (legacy_new[column].isna() & old_panel[column].isna()))
    legacy_comparison_rows.append({'column': column, 'comparison': 'exact', 'mismatch_rows': int(mismatch.sum())})
for column in ['night_share', 'weekend_share']:
    match = np.isclose(legacy_new[column], old_panel[column], rtol=1e-12, atol=1e-12, equal_nan=True)
    if all_raw_identical:
        assert match.all()
    legacy_comparison_rows.append({'column': column, 'comparison': 'rtol_atol_1e-12', 'mismatch_rows': int((~match).sum())})
old_exposure_rows = old_performance.sort_values(KEY).reset_index(drop=True)
pd.testing.assert_frame_equal(legacy_new[KEY + EXPOSURES], old_exposure_rows[KEY + EXPOSURES],
                              check_dtype=False, check_exact=True)
legacy_comparison = pd.DataFrame(legacy_comparison_rows)
if all_raw_identical:
    assert legacy_comparison['mismatch_rows'].eq(0).all()
legacy_comparison['identical_source_hashes'] = all_raw_identical
display(legacy_comparison)

**해석:** 해시가 같은 원자료에서는 기존 2종 값의 일치를 요구한다. 원본이 개정된 경우 표의 차이는 원자료 수정과 연결될 수 있으며 신규 패널은 최신 검증된 값을 유지한다. 공연 노출은 별도 최신 스냅샷이므로 동결 기준선의 과거 노출과 동일할 필요가 없다.

## 11. 최종 검증과 저장
완전한 격자, 선택 업종, 미제공 매출 보존, 비율 범위, 원본 개정 여부를 검사한다. 현재 경계와 시설 좌표를 모든 과거 분기에 공통 적용한다는 한계는 검사 통과와 별개다.

In [ ]:
source_unchanged = all(sha256_file(project_path(row.path)) == row.sha256 for row in source_hashes.itertuples())
checks = [
    ('source_total_equals_regular_plus_franchise', bool(store_identity.all()), len(stores)),
    ('panel_total_equals_regular_plus_franchise', panel.loc[observed_stores, 'total_store_count'].eq(panel.loc[observed_stores, 'regular_store_count'] + panel.loc[observed_stores, 'franchise_store_count']).all(), int(observed_stores.sum())),
    ('positive_sales_zero_regular_has_franchise', panel.loc[panel['positive_sales_zero_regular_stores'], 'franchise_store_count'].gt(0).all(), int(panel['positive_sales_zero_regular_stores'].sum())),
    ('no_positive_sales_with_zero_total_stores', not panel['positive_sales_zero_stores'].any(), int(panel['positive_sales_zero_stores'].sum())),
    ('grid_rows_18120', len(panel) == 18120, len(panel)),
    ('unique_panel_key', not panel.duplicated(KEY).any(), int(panel.duplicated(KEY).sum())),
    ('all_official_food_codes_10', set(panel['industry_id']) == set(selected_ids), panel['industry_id'].nunique()),
    ('151_areas_12_quarters', panel['area_id'].nunique() == 151 and panel['quarter'].nunique() == 12, '151 × 12'),
    ('food_labels_stable_6_sources', food_source.groupby('industry_id')['industry'].nunique().eq(1).all(), 6),
    ('missing_sales_not_imputed', panel.loc[~panel['sales_record_present'], 'total_sales'].isna().all(), int(panel['sales_record_missing'].sum())),
    ('missing_stores_not_imputed', panel.loc[~panel['store_record_present'], 'store_count'].isna().all(), int(panel['store_record_missing'].sum())),
    ('night_share_in_0_1', panel['night_share'].dropna().between(0, 1).all(), int(panel['night_share'].notna().sum())),
    ('weekend_share_in_0_1', panel['weekend_share'].dropna().between(0, 1).all(), int(panel['weekend_share'].notna().sum())),
    ('legacy_values_exact_if_same_sources', (not all_raw_identical) or legacy_comparison['mismatch_rows'].eq(0).all(), f'same_source_hashes={all_raw_identical}'),
    ('exposure_duplicates_agree', exposure_agreement.eq(1).all().all(), len(exposure_agreement)),
    ('exposure_complete', panel[EXPOSURES].notna().all().all(), len(exposures)),
    ('exposure_nested_radii', (panel[EXPOSURES[0]].le(panel[EXPOSURES[1]]) & panel[EXPOSURES[1]].le(panel[EXPOSURES[2]])).all(), '300 ≤ 500 ≤ 800'),
    ('source_files_unchanged', source_unchanged, len(source_hashes))]
validation = pd.DataFrame(checks, columns=['check', 'passed', 'evidence'])
assert validation['passed'].all(), validation.loc[~validation['passed']].to_dict('records')
display(validation)

In [ ]:
panel = panel.sort_values(KEY).reset_index(drop=True)
outputs = {
    ANALYSIS / 'food_industry_catalog.csv': catalog,
    ANALYSIS / 'food_industry_panel.csv': panel,
    REPORTS / '08_source_industry_catalog.csv': source_catalog,
    REPORTS / '08_source_quality.csv': source_quality,
    REPORTS / '08_source_revisions.csv': source_revisions,
    REPORTS / '08_code_label_consistency.csv': label_check.reset_index(),
    REPORTS / '08_source_fingerprints.csv': source_hashes,
    REPORTS / '08_industry_coverage.csv': coverage,
    REPORTS / '08_district_industry_coverage.csv': district_coverage,
    REPORTS / '08_quarter_industry_coverage.csv': quarter_coverage,
    REPORTS / '08_value_flags.csv': flag_summary,
    REPORTS / '08_flagged_panel_rows.csv': panel.loc[panel[flag_columns].any(axis=1)],
    REPORTS / '08_legacy_two_industry_comparison.csv': legacy_comparison,
    REPORTS / '08_validation_checks.csv': validation}
for path, frame in outputs.items():
    frame.to_csv(path, index=False, encoding='utf-8-sig')
summary = {'created_utc': datetime.now(timezone.utc).isoformat(),
    'store_count_definitions': {'regular_store_count': '일반점포 수', 'franchise_store_count': '프랜차이즈점포 수', 'total_store_count': '일반+프랜차이즈 전체점포 수'},
    'deprecated_store_aliases': {'store_count': 'regular_store_count', 'franchise_stores': 'franchise_store_count', 'similar_industry_store_count': 'total_store_count'},
    'sales_per_store_population_validated': False, 'sales_per_store_included': False,
    'common_sample_before_model_filters': common_sample_summary.to_dict('records')[0],
    'sales_coverage_by_total_store_size': store_size_coverage.reset_index().to_dict('records'),
    'boundary_note': seoul_manifest.get('boundary_note') if seoul_manifest else 'current_boundaries_for_all_quarters',
    'areas': 151, 'quarters': QUARTERS, 'industries': industry_table[['industry_id', 'industry']].to_dict('records'),
    'panel_sha256': sha256_file(ANALYSIS / 'food_industry_panel.csv'),
    'catalog_sha256': sha256_file(ANALYSIS / 'food_industry_catalog.csv'),
    'panel_rows': len(panel), 'observed_sales_rows': int(panel['total_sales'].notna().sum()),
    'sales_record_absent_rows': int(panel['sales_record_missing'].sum()),
    'observed_store_rows': int(panel['store_count'].notna().sum()),
    'selection_rule': 'all_CS100_food_services_fixed_before_models; no retail; no p-value selection',
    'legacy_two_industry_exact_comparison': 'passed' if all_raw_identical else 'revised_sources_differences_reported',
    'seoul_refresh_run_id': seoul_manifest['run_id'] if seoul_manifest else None,
    'raw_sources_identical_to_baseline': all_raw_identical,
    'legacy_comparison_mismatch_cells': int(legacy_comparison['mismatch_rows'].sum()), 'validation_checks_passed': int(validation['passed'].sum()),
    'exposure_source': 'dataset/analysis/final/performance_sales_panel.csv',
    'exposure_linkage_coverage': exposure_linkage_coverage.to_dict('records'),
    'exposure_snapshot_linkage': linkage_summary,
    'limitations': ['quarterly_not_daily', 'store_count_is_deprecated_regular_store_alias', 'sales_per_store_excluded_unverified_population_match', 'selective_sales_missingness_and_common_sample_comparability', 'no_missing_sales_zero_imputation',
        'official_ID_vs_provisional_name_linkage',
        ('coordinates_incomplete' if linkage_summary['performance_ids_with_coordinates'] < linkage_summary['performance_ids']
         else 'current_list_coordinates_complete_historical_locations_unverified'),
        'current_boundaries_and_facility_locations_for_all_quarters',
        'overlapping_areas_preclude_unique_economic_total', 'validated_local_snapshot_not_live_network', 'seoul_2024_spatial_unit_revision_and_current_boundary_limit'],
    'source_files': source_hashes.to_dict('records'),
    'python': platform.python_version(),
    'packages': {name: importlib.metadata.version(name) for name in ['pandas', 'numpy', 'matplotlib', 'nbformat']}}
(REPORTS / '08_food_scope_metadata.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
display(pd.DataFrame([{'상권': 151, '업종': 10, '분기': 12, '격자 행': len(panel),
    '매출 관측 행': summary['observed_sales_rows'], '점포 관측 행': summary['observed_store_rows'],
    '통과 검증': int(validation['passed'].sum())}]))

**후속 입력:** `dataset/analysis/final/food_industry_panel.csv`와 `food_industry_catalog.csv`를 05~08에서 사용한다. 모든 업종을 유지하고 자료 제공률·관측 상권·분기를 함께 보고한다. 보고서 파일의 08 접두어는 기존 결과와의 호환을 위한 산출물 이름이며 현재 실행 순서는 04다.